# EA1 - Actividad 1.1: Introduccion a Apache Spark

## Objetivos
- Entender que es Apache Spark y por que es relevante en Big Data
- Crear y configurar una SparkSession
- Leer y explorar datos basicos desde un archivo CSV
- Realizar operaciones fundamentales: select, filter, groupBy

## Conceptos Clave

### Que es Apache Spark?

Apache Spark es un **motor de procesamiento distribuido** de datos a gran escala, disenado para ser rapido y de proposito general.

**Breve historia:**
- Nacio en 2009 en el AMPLab de UC Berkeley
- Donado a la Apache Software Foundation en 2013
- Hoy es uno de los proyectos open-source mas activos en Big Data

**Ventajas sobre MapReduce tradicional:**
- **Velocidad:** Procesamiento en memoria (hasta 100x mas rapido que MapReduce en disco)
- **Facilidad de uso:** APIs en Python, Scala, Java y R
- **Versatilidad:** Batch, streaming, ML, grafos y SQL en un solo framework
- **Evaluacion lazy:** Las transformaciones no se ejecutan hasta que se necesita un resultado

**Ecosistema Spark:**

| Componente | Descripcion |
|------------|-------------|
| Spark Core | Motor base de procesamiento distribuido |
| Spark SQL | Consultas estructuradas y DataFrames |
| Spark Streaming | Procesamiento de datos en tiempo real |
| MLlib | Libreria de Machine Learning distribuido |
| GraphX | Procesamiento de grafos |

## Arquitectura de Spark

```
                    +-------------------+
                    |   Aplicacion      |
                    |   (tu codigo)     |
                    +--------+----------+
                             |
                    +--------v----------+
                    |   SparkSession    |
                    |   (Driver)        |
                    +--------+----------+
                             |
                    +--------v----------+
                    | Cluster Manager   |
                    | (YARN/Mesos/      |
                    |  Standalone)      |
                    +---+----------+----+
                        |          |
               +--------v--+  +----v-------+
               | Executor 1|  | Executor 2 |
               | [Task]    |  | [Task]     |
               | [Task]    |  | [Task]     |
               +-----------+  +------------+
```

- **Driver:** Programa principal que coordina la ejecucion
- **SparkSession:** Punto de entrada unificado (reemplaza SparkContext, SQLContext, HiveContext)
- **Cluster Manager:** Asigna recursos (YARN, Mesos, Kubernetes o Standalone)
- **Executors:** Procesos que ejecutan las tareas en los nodos del cluster

In [12]:
# Setup: Crear SparkSession
# La SparkSession es el punto de entrada principal para trabajar con Spark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("01_introduccion_spark") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 3.5.0


## Leer datos desde un archivo CSV

Spark puede leer datos de multiples formatos: CSV, JSON, Parquet, ORC, JDBC, etc.

Parametros importantes de `spark.read.csv()`:
- `header=True`: La primera fila contiene los nombres de las columnas
- `inferSchema=True`: Spark infiere automaticamente los tipos de datos

In [13]:
# Leer el archivo flights.csv
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# Verificamos que se leyo correctamente
print(f"Tipo del objeto: {type(df)}")

Tipo del objeto: <class 'pyspark.sql.dataframe.DataFrame'>


## Explorar los datos

Una vez cargado el DataFrame, tenemos varias formas de explorarlo.

In [14]:
# Mostrar las primeras 5 filas del DataFrame
df.show(5)

+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute|
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|2014|    1|  1|       1|       96|     235|       70|     AS| N508AS|   145|   PDX| ANC|     194|    1542|   0|     1|
|2014|    1|  1|       4|       -6|     738|      -23|     US| N195UW|  1830|   SEA| CLT|     252|    2279|   0|     4|
|2014|    1|  1|       8|       13|     548|       -4|     UA| N37422|  1609|   PDX| IAH|     201|    1825|   0|     8|
|2014|    1|  1|      28|       -2|     800|      -23|     US| N547UW|   466|   PDX| CLT|     251|    2282|   0|    28|
|2014|    1|  1|      34|       44|     325|       43|     AS| N762AS|   121|   SEA| ANC|     201|    1448|   0|    34|
+----+-----+---+--------+---------+-----

In [15]:
# Ver el esquema (nombres de columnas y tipos de datos)
df.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- dep_time: string (nullable = true)
 |-- dep_delay: string (nullable = true)
 |-- arr_time: string (nullable = true)
 |-- arr_delay: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- tailnum: string (nullable = true)
 |-- flight: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- dest: string (nullable = true)
 |-- air_time: string (nullable = true)
 |-- distance: integer (nullable = true)
 |-- hour: string (nullable = true)
 |-- minute: string (nullable = true)



In [16]:
# Contar el numero total de filas
print(f"Numero total de filas: {df.count()}")

# Ver los nombres de las columnas
print(f"Columnas: {df.columns}")

Numero total de filas: 162049
Columnas: ['year', 'month', 'day', 'dep_time', 'dep_delay', 'arr_time', 'arr_delay', 'carrier', 'tailnum', 'flight', 'origin', 'dest', 'air_time', 'distance', 'hour', 'minute']


In [17]:
# Estadisticas descriptivas basicas (media, desviacion estandar, min, max, count)
df.describe().show()

+-------+------+-----------------+------------------+-----------------+-----------------+------------------+------------------+-------+-------+------------------+------+------+-----------------+------------------+------------------+------------------+
|summary|  year|            month|               day|         dep_time|        dep_delay|          arr_time|         arr_delay|carrier|tailnum|            flight|origin|  dest|         air_time|          distance|              hour|            minute|
+-------+------+-----------------+------------------+-----------------+-----------------+------------------+------------------+-------+-------+------------------+------+------+-----------------+------------------+------------------+------------------+
|  count|162049|           162049|            162049|           162049|           162049|            162049|            162049| 162049| 161801|            162049|162049|162049|           162049|            162049|            162049|            

## Seleccionar columnas con `select()`

El metodo `select()` permite elegir columnas especificas del DataFrame, similar a SELECT en SQL.

In [19]:
# Setup: Crear SparkSession
# La SparkSession es el punto de entrada principal para trabajar con Spark

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("01_introduccion_spark") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 3.5.0


In [21]:
# Leer el archivo flights.csv
df = spark.read.csv("/home/jovyan/datos/flights.csv", header=True, inferSchema=True)

# Verificamos que se leyo correctamente
print(f"Tipo del objeto: {type(df)}")

Tipo del objeto: <class 'pyspark.sql.dataframe.DataFrame'>


In [24]:
df.select("carrier", "dep_delay", "arr_delay").show(5)

+-------+---------+---------+
|carrier|dep_delay|arr_delay|
+-------+---------+---------+
|     AS|       96|       70|
|     US|       -6|      -23|
|     UA|       13|       -4|
|     US|       -2|      -23|
|     AS|       44|       43|
+-------+---------+---------+
only showing top 5 rows



## Filtrar datos con `filter()`

El metodo `filter()` permite seleccionar filas que cumplan una condicion, similar a WHERE en SQL.

In [25]:
# Ejemplo: filtrar vuelos con distancia mayor a 2000 millas
vuelos_largos = df.filter(df["DISTANCE"] > 2000)
print(f"Vuelos de larga distancia (>2000 mi): {vuelos_largos.count()}")
vuelos_largos.show(5)

Vuelos de larga distancia (>2000 mi): 23707
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute|
+----+-----+---+--------+---------+--------+---------+-------+-------+------+------+----+--------+--------+----+------+
|2014|    1|  1|       4|       -6|     738|      -23|     US| N195UW|  1830|   SEA| CLT|     252|    2279|   0|     4|
|2014|    1|  1|      28|       -2|     800|      -23|     US| N547UW|   466|   PDX| CLT|     251|    2282|   0|    28|
|2014|    1|  1|     536|        1|    1334|       -6|     UA| N574UA|   478|   SEA| EWR|     268|    2402|   5|    36|
|2014|    1|  1|     622|        2|    1412|      -19|     UA| N36472|  1021|   PDX| EWR|     270|    2434|   6|    22|
|2014|    1|  1|     624|       -6|    1401|       -6|     DL| N617DL|   968|   SEA| ATL|     235|    2182|   6|    

## Agrupar datos con `groupBy()`

El metodo `groupBy()` agrupa los datos por una o mas columnas y permite aplicar funciones de agregacion como `count()`, `sum()`, `avg()`, etc.

In [26]:
# Ejemplo: contar vuelos por mes
df.groupBy("MONTH").count().orderBy("MONTH").show()

+-----+-----+
|MONTH|count|
+-----+-----+
|    1|12155|
|    2|10889|
|    3|12923|
|    4|13048|
|    5|14035|
|    6|14966|
|    7|15803|
|    8|16059|
|    9|13504|
|   10|13377|
|   11|12406|
|   12|12884|
+-----+-----+



---
## Ejercicios

Ahora es tu turno de practicar. Completa los siguientes ejercicios.

In [29]:
# =============================================================
# EJERCICIO 1: Seleccionar columnas especificas
# =============================================================
# TODO: Usa df.select() para seleccionar las columnas:
#   "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DEPARTURE_DELAY"
# Luego muestra las primeras 10 filas con .show(10)
#
# Pista: df.select("col1", "col2", "col3").show(n)
#year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute
# Escribe tu codigo aqui:

df.select("origin", "dest", "dep_delay").show(5)

+------+----+---------+
|origin|dest|dep_delay|
+------+----+---------+
|   PDX| ANC|       96|
|   SEA| CLT|       -6|
|   PDX| IAH|       13|
|   PDX| CLT|       -2|
|   SEA| ANC|       44|
+------+----+---------+
only showing top 5 rows



In [31]:
# =============================================================
# EJERCICIO 2: Filtrar vuelos con retraso
# =============================================================
# TODO: Usa df.filter() para obtener solo los vuelos que tienen
#   DEPARTURE_DELAY > 0 (es decir, vuelos que salieron con retraso).
# Luego cuenta cuantos vuelos con retraso hay usando .count()
#
# Pista: df.filter(df["columna"] > valor).count()
#year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute
# Escribe tu codigo aqui:
df.filter(df["dep_delay"] > 0).count()

56538

In [33]:
# =============================================================
# EJERCICIO 3: Contar vuelos por aerolinea
# =============================================================
# TODO: Usa df.groupBy("AIRLINE").count() para contar la cantidad
#   de vuelos por cada aerolinea.
# Luego ordena los resultados de mayor a menor con
#   .orderBy("count", ascending=False)
# Finalmente muestra el resultado con .show()
#year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute
# Pista: df.groupBy("col").count().orderBy("count", ascending=False).show()

# Escribe tu codigo aqui:
df.groupBy("carrier").count().orderBy("count", ascending=False).show()

+-------+-----+
|carrier|count|
+-------+-----+
|     AS|62460|
|     WN|23355|
|     OO|18710|
|     DL|16716|
|     UA|16671|
|     AA| 7586|
|     US| 5946|
|     B6| 3540|
|     VX| 3272|
|     F9| 2698|
|     HA| 1095|
+-------+-----+



---
## Resumen

En esta actividad aprendimos:

1. **Que es Apache Spark:** Un motor de procesamiento distribuido, rapido y versatil
2. **Arquitectura:** Driver, Executors, Cluster Manager y SparkSession como punto de entrada
3. **Crear SparkSession:** `SparkSession.builder.appName(...).master(...).getOrCreate()`
4. **Leer CSV:** `spark.read.csv(path, header=True, inferSchema=True)`
5. **Explorar datos:** `.show()`, `.printSchema()`, `.count()`, `.columns`, `.describe()`
6. **Operaciones basicas:** `.select()`, `.filter()`, `.groupBy()`, `.orderBy()`

---
## Desafio Extra (Opcional)

**Para estudiantes avanzados:**

Encuentra el **top 5 de rutas (origen-destino) mas frecuentes**.

Pista: Puedes agrupar por dos columnas a la vez con `groupBy("col1", "col2")`.

In [34]:
# =============================================================
# DESAFIO: Top 5 rutas mas frecuentes
# =============================================================
# TODO: Agrupa por ORIGIN_AIRPORT y DESTINATION_AIRPORT,
#   cuenta las ocurrencias, ordena de mayor a menor
#   y muestra solo las 5 primeras rutas.
#year|month|day|dep_time|dep_delay|arr_time|arr_delay|carrier|tailnum|flight|origin|dest|air_time|distance|hour|minute
# Pista: df.groupBy("col1", "col2").count().orderBy(...).show(5)

# Escribe tu codigo aqui:
df.groupBy("origin", "dest").count().orderBy("count", ascending=False).show(5)

+------+----+-----+
|origin|dest|count|
+------+----+-----+
|   SEA| SFO| 7630|
|   SEA| LAX| 7455|
|   SEA| ANC| 6149|
|   SEA| LAS| 5732|
|   SEA| DEN| 5578|
+------+----+-----+
only showing top 5 rows



In [35]:
# Detener la SparkSession al finalizar
spark.stop()
print("SparkSession detenida correctamente.")

SparkSession detenida correctamente.
